<a href="https://colab.research.google.com/github/hamzahfarooque1995/MOE_Repo/blob/main/ApproxMoE_Hamzah_Farooq_Thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ApproxMoE: Three-Tier Mixture-of-Experts for Bearing Fault Diagnosis

This notebook is a clean, self-contained proof-of-concept implementation
of the ApproxMoE system described in the thesis.

**What this notebook builds, in order:**
1. Downloads and preprocesses the CWRU and XJTU-SY bearing datasets.
2. Trains a full-precision baseline CNN on CWRU.
3. Builds the three-tier difficulty gate (Easy / Medium / Hard), using
   noise-based entropy scoring rather than raw confidence.
4. Trains three experts at different numeric precisions:
   - Expert A: 4-bit, handles Easy signals.
   - Expert B: 8-bit, handles Medium signals.
   - Expert C: mixed precision (INT8 weights, FP32 activations), trained
     on both the hardest CWRU signals and the XJTU-SY dataset, so it
     handles Hard in-domain signals *and* out-of-domain signals.
5. Assembles the full ApproxMoE system: an out-of-distribution check that
   always wins, a three-way tier decision, and a confidence cascade that
   can escalate a signal from A to B to C if the assigned expert isn't
   confident.
6. Evaluates everything and shows the results in confusion matrix form

## Phase 0 -- Setup

In [ ]:
#0-0
# Install packages this notebook needs that aren't in Colab by default.
!pip install gdown torchaudio scipy scikit-learn seaborn tqdm joblib -q


In [ ]:
#0-1
# Mount Google Drive so trained models and processed data persist across
# Colab sessions. Without this, everything is lost when the runtime resets.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#0-2
# All paths and hyperparameters live here so the whole notebook can be
# tuned from one place.
import os, numpy as np, torch

# -- Paths --------------------------------------------------------------
# CWRU_RAW_DIR / XJTU_DATASET_PATH: where the original downloaded data goes
# CWRU_PROC_DIR / XJTU_SPEC_PATH: where we cache processed arrays so we
#   never have to redo expensive preprocessing on a second run
CWRU_RAW_DIR      = "/content/CWRU/cwru"
CWRU_PROC_DIR     = "/content/CWRU/processed"
XJTU_DATASET_PATH = "/content/drive/MyDrive/XJTU_dataset"
XJTU_SPEC_PATH    = "/content/drive/MyDrive/XJTU_spectrogram"
BEARING_DIR       = "/content/drive/MyDrive/bearing_model"
MODEL_SAVE_PATH   = os.path.join(BEARING_DIR, "best_model.pth")
GATE_SAVE_PATH    = os.path.join(BEARING_DIR, "gate_model.pth")

for d in [CWRU_PROC_DIR, BEARING_DIR, XJTU_DATASET_PATH, XJTU_SPEC_PATH]:
    os.makedirs(d, exist_ok=True)

# -- Signal preprocessing settings --------------------------------------
WINDOW_SIZE = 2048         # samples per window (~171ms at 12kHz)
OVERLAP     = 0.5          # 50% overlap between consecutive windows
CWRU_FS     = 12000        # CWRU sampling rate (Hz)
XJTU_FS     = 25600        # XJTU-SY sampling rate (Hz), resampled to CWRU_FS
N_FFT       = 256          # STFT window size
HOP_LENGTH  = 64           # STFT hop length (75% overlap)
SPEC_SIZE   = 128          # spectrograms are resized to SPEC_SIZE x SPEC_SIZE
NUM_CLASSES = 3            # Normal, Inner Race, Outer Race

# -- Difficulty scoring settings (see Phase 8) ---------------------------
# We measure difficulty by adding noise and checking how confused the
# baseline model gets. SNR_DB=0 means the noise is as strong as the
# signal itself -- deliberately harsh, so the resulting scores actually
# spread out instead of clustering near "always confident".
SNR_DB    = 0.0
N_REPEATS = 8   # average over this many independent noisy copies per window

CLASS_NAMES = {0: "Normal", 1: "Inner Race", 2: "Outer Race"}
TIER_NAMES  = {0: "Easy", 1: "Medium", 2: "Hard"}

print("Configuration loaded.")


## Phase 1 -- CWRU: Download and Build Raw Windows

In [ ]:
#1-0
# Download the CWRU dataset if it isn't already on disk.
import gdown, zipfile

CWRU_ZIP_ID = "18aq4iY93ssuZHsh3zIlL09oWHiVe2U-m"

if not os.path.exists(CWRU_RAW_DIR) or not os.listdir(CWRU_RAW_DIR):
    print("Downloading CWRU dataset...")
    zip_out = "/content/cwrudata.zip"
    gdown.download(f"https://drive.google.com/uc?id={CWRU_ZIP_ID}", zip_out, quiet=False)
    with zipfile.ZipFile(zip_out, 'r') as z:
        z.extractall("/content/CWRU")
    print("Extraction complete.")
else:
    print("CWRU dataset already present -- skipping download.")


In [ ]:
#1-1
# Helper functions for turning raw CWRU .mat files into fixed-length,
# normalised 1-D windows.
import scipy.io as sio
from scipy.stats import kurtosis as scipy_kurtosis, skew as scipy_skew
from sklearn.model_selection import train_test_split
from tqdm import tqdm


def load_drive_end_signal(filepath: str) -> np.ndarray:
    """Read the drive-end accelerometer channel out of a CWRU .mat file."""
    data = sio.loadmat(filepath)
    for key in data.keys():
        if "_DE_time" in key:
            return data[key].squeeze()
    raise ValueError(f"Drive-end signal not found in {filepath}")


def get_fault_label(filename: str) -> int:
    """Work out the fault class from a CWRU filename. Ball Fault (label 1)
    is filtered out later since XJTU-SY has no equivalent class."""
    f = filename.upper()
    if   f.startswith("NORMAL"): return 0
    elif f.startswith("B"):      return 1   # Ball Fault -- removed later
    elif f.startswith("IR"):     return 2
    elif f.startswith("OR"):     return 3
    else: raise ValueError(f"Cannot infer label from: {filename}")


def segment_signal(signal: np.ndarray, window: int = WINDOW_SIZE,
                    overlap: float = OVERLAP) -> np.ndarray:
    """Cut a long 1-D signal into fixed-length overlapping windows."""
    stride = int(window * (1 - overlap))
    return np.array([signal[i:i + window]
                     for i in range(0, len(signal) - window, stride)])


def normalize_window(w: np.ndarray) -> np.ndarray:
    """Zero-mean, unit-variance normalisation of one window."""
    return (w - w.mean()) / (w.std() + 1e-8)


print("Phase 1 helper functions defined.")


In [ ]:
#1-2
# Build the raw windowed dataset once, then cache it to disk. Also removes
# Ball Fault samples and remaps labels to the shared 3-class space
# (0=Normal, 1=Inner Race, 2=Outer Race) that both CWRU and XJTU-SY use.
_x_path = os.path.join(CWRU_PROC_DIR, "X_train.npy")

if not os.path.exists(_x_path):
    X_all, y_all = [], []
    mat_files = [f for f in os.listdir(CWRU_RAW_DIR) if f.endswith(".mat")]
    print(f"Found {len(mat_files)} .mat files -- building windows...")

    for fname in tqdm(mat_files, desc="Loading CWRU"):
        label  = get_fault_label(fname)
        signal = load_drive_end_signal(os.path.join(CWRU_RAW_DIR, fname))
        wins   = segment_signal(signal)
        wins   = np.array([normalize_window(w) for w in wins])
        X_all.extend(wins); y_all.extend([label] * len(wins))

    X_all = np.array(X_all); y_all = np.array(y_all)

    # Drop Ball Fault (label 1) -- XJTU-SY has no equivalent class.
    keep = y_all != 1
    X_all, y_all = X_all[keep], y_all[keep]
    # Remap remaining labels {0, 2, 3} -> {0, 1, 2}
    remap = {0: 0, 2: 1, 3: 2}
    y_all = np.array([remap[v] for v in y_all])

    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)

    for name, arr in [("X_train", X_train), ("X_test", X_test),
                       ("y_train", y_train), ("y_test", y_test)]:
        np.save(os.path.join(CWRU_PROC_DIR, f"{name}.npy"), arr)
    print("Saved raw windows to disk.")
else:
    X_train = np.load(os.path.join(CWRU_PROC_DIR, "X_train.npy"))
    X_test  = np.load(os.path.join(CWRU_PROC_DIR, "X_test.npy"))
    y_train = np.load(os.path.join(CWRU_PROC_DIR, "y_train.npy"))
    y_test  = np.load(os.path.join(CWRU_PROC_DIR, "y_test.npy"))
    print("Loaded raw windows from disk.")

print(f"Train: {X_train.shape}  labels: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test:  {X_test.shape}  labels: {dict(zip(*np.unique(y_test, return_counts=True)))}")


## Phase 2 -- CWRU: Convert Windows to Spectrograms

In [ ]:
#2-0
# Turn each 1-D raw window into a 2-D time-frequency image (spectrogram),
# which is what the CNN actually sees. This is done once and cached.
import torch, torchaudio
import torch.nn.functional as F

_spec_path = os.path.join(CWRU_PROC_DIR, "X_train_spec.npy")
_stft = torchaudio.transforms.Spectrogram(n_fft=N_FFT, hop_length=HOP_LENGTH, power=2)


def window_to_spectrogram(w: np.ndarray) -> np.ndarray:
    """1-D window -> 2-D log-power spectrogram."""
    t = torch.tensor(w, dtype=torch.float32)
    return torch.log(_stft(t) + 1e-8).numpy()


def batch_to_spectrograms(X: np.ndarray, desc: str = "") -> np.ndarray:
    return np.array([window_to_spectrogram(w) for w in tqdm(X, desc=desc)])


def resize_specs(X: np.ndarray) -> np.ndarray:
    """Resize every spectrogram to a fixed SPEC_SIZE x SPEC_SIZE image."""
    t = torch.tensor(X).unsqueeze(1)
    r = F.interpolate(t, size=(SPEC_SIZE, SPEC_SIZE), mode='bilinear', align_corners=False)
    return r.squeeze(1).numpy()


if not os.path.exists(_spec_path):
    print("Converting windows to spectrograms (this can take a few minutes)...")
    X_tr_spec = resize_specs(batch_to_spectrograms(X_train, "Train"))
    X_te_spec = resize_specs(batch_to_spectrograms(X_test,  "Test"))

    # Normalise using training-set statistics so the model sees a
    # consistent numeric range. spec_mean/spec_std are saved so the exact
    # same normalisation can be applied later to XJTU-SY spectrograms.
    spec_mean = float(X_tr_spec.mean())
    spec_std  = float(X_tr_spec.std())
    X_tr_spec = (X_tr_spec - spec_mean) / spec_std
    X_te_spec = (X_te_spec - spec_mean) / spec_std

    np.save(os.path.join(CWRU_PROC_DIR, "X_train_spec.npy"), X_tr_spec)
    np.save(os.path.join(CWRU_PROC_DIR, "X_test_spec.npy"),  X_te_spec)
    np.save(os.path.join(CWRU_PROC_DIR, "spec_mean.npy"), np.array([spec_mean]))
    np.save(os.path.join(CWRU_PROC_DIR, "spec_std.npy"),  np.array([spec_std]))
    print(f"Saved spectrograms. mean={spec_mean:.4f} std={spec_std:.4f}")
else:
    X_tr_spec = np.load(os.path.join(CWRU_PROC_DIR, "X_train_spec.npy"))
    X_te_spec = np.load(os.path.join(CWRU_PROC_DIR, "X_test_spec.npy"))
    spec_mean = float(np.load(os.path.join(CWRU_PROC_DIR, "spec_mean.npy"))[0])
    spec_std  = float(np.load(os.path.join(CWRU_PROC_DIR, "spec_std.npy"))[0])
    print(f"Loaded spectrograms from disk. mean={spec_mean:.4f} std={spec_std:.4f}")

print(f"Train spec: {X_tr_spec.shape}  Test spec: {X_te_spec.shape}")


## Phase 3 -- Baseline CNN

In [ ]:
#3-0
# The network every expert is based on. Three conv layers followed by
# three fully-connected layers, no pooling or batch norm -- kept simple
# so quantization (later phases) behaves predictably.
import torch.nn as nn

class SpectrogramCNN(nn.Module):
    """Backbone shared by the baseline model and all three experts.
    They differ only in numeric precision and training data, never in
    architecture."""
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=1)
        self.conv3 = nn.Conv2d(64,  1, kernel_size=1)
        self.fc1   = nn.Linear(SPEC_SIZE * SPEC_SIZE, 256)
        self.fc2   = nn.Linear(256, 256)
        self.fc3   = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


_dummy = torch.randn(4, 1, SPEC_SIZE, SPEC_SIZE)
assert SpectrogramCNN()(_dummy).shape == (4, NUM_CLASSES)
total_params = sum(p.numel() for p in SpectrogramCNN().parameters())
print(f"SpectrogramCNN -- {total_params:,} parameters. Forward pass OK.")


In [ ]:
#3-1
# Wraps numpy spectrogram arrays into a PyTorch Dataset.
from torch.utils.data import Dataset, DataLoader

class SpectrogramDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # add channel dim
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_loader = DataLoader(SpectrogramDataset(X_tr_spec, y_train),
                          batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SpectrogramDataset(X_te_spec, y_test),
                          batch_size=32, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
#3-2
# Standard training / evaluation utilities used by the baseline model
# and reused later for the within-XJTU initialisation model.
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

model = SpectrogramCNN().to(device)

# Class-weighted loss keeps any minor class imbalance from dominating training.
_cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
_cw_tensor = torch.tensor(_cw, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=_cw_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def train_one_epoch(mdl, ldr, opt, crit):
    mdl.train(); total = 0.0
    for x, y in ldr:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(mdl(x), y)
        loss.backward(); opt.step()
        total += loss.item()
    return total / len(ldr)


def evaluate(mdl, ldr):
    """Returns (accuracy, macro_f1, predictions, true_labels)."""
    mdl.eval(); preds, targets = [], []
    with torch.no_grad():
        for x, y in ldr:
            p = torch.argmax(mdl(x.to(device)), dim=1)
            preds.extend(p.cpu().numpy()); targets.extend(y.numpy())
    return (accuracy_score(targets, preds),
            f1_score(targets, preds, average='macro', zero_division=0),
            preds, targets)


print(f"Class weights: {[round(float(w), 3) for w in _cw]}")
print("Training utilities ready.")


In [ ]:
#3-3
# Train the baseline CNN, keeping the checkpoint with the best test F1.
if os.path.exists(MODEL_SAVE_PATH):
    print("Baseline model already trained -- loading from disk.")
    ckpt = torch.load(MODEL_SAVE_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded. Best F1 = {ckpt['best_f1']:.4f}")
else:
    EPOCHS, PATIENCE = 30, 5
    best_f1, patience_ctr = 0.0, 0

    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, criterion)
        acc, f1, _, _ = evaluate(model, test_loader)
        print(f"Epoch {epoch:02d}/{EPOCHS}  loss={loss:.4f}  acc={acc:.4f}  F1={f1:.4f}", end="")
        if f1 > best_f1:
            best_f1, patience_ctr = f1, 0
            torch.save({"model_state_dict": model.state_dict(), "best_f1": best_f1}, MODEL_SAVE_PATH)
            print("  <- best", end="")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch}."); break
        print()

    # Reload the best checkpoint (not necessarily the last epoch trained).
    ckpt = torch.load(MODEL_SAVE_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"\nBest F1 = {ckpt['best_f1']:.4f}")


In [ ]:
#3-4
# Confusion matrix for the baseline model on the CWRU test set.
acc, f1, preds, targets = evaluate(model, test_loader)
print(classification_report(targets, preds, target_names=list(CLASS_NAMES.values())))

cm      = confusion_matrix(targets, preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
kw = dict(xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values())
sns.heatmap(cm,      annot=True, fmt="d",   cmap="Blues",  ax=axes[0], **kw)
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Greens", ax=axes[1], **kw)
for ax, t in zip(axes, ["Counts", "% per row"]):
    ax.set_title(f"Confusion Matrix ({t})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle(f"Baseline CNN on CWRU  \u2014  Acc={acc:.1%}  F1={f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig("conf_cwru.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 4 -- XJTU-SY: Download, Preprocess, Zero-Shot Baseline

In [ ]:
#4-0
# Download the XJTU-SY dataset if it isn't already on disk.
if not os.path.exists(XJTU_DATASET_PATH) or not os.listdir(XJTU_DATASET_PATH):
    print("Downloading XJTU-SY dataset...")
    XJTU_FOLDER_URL = "https://drive.google.com/drive/folders/1XEgLU2DBYmaGbb4h3WIETCfYD84_7dIW?usp=sharing"
    gdown.download_folder(XJTU_FOLDER_URL, output=XJTU_DATASET_PATH, quiet=False)
    print("Download complete.")
else:
    print("XJTU-SY dataset already present -- skipping download.")


In [ ]:
#4-1
# Which XJTU-SY bearings we use, and what fault each one represents.
# Bearing1_1 and Bearing3_1 are excluded: their RMS trajectories show they
# were already degraded at the very start of recording, so the standard
# "first 10% = healthy" labelling assumption doesn't hold for them.
import pandas as pd
from scipy.signal import resample_poly

XJTU_BEARING_MAP = {
    "37.5Hz11kN/Bearing2_1": 1,   # Inner Race
    "37.5Hz11kN/Bearing2_5": 2,   # Outer Race
    "40Hz10kN/Bearing3_3":   1,   # Inner Race
    "40Hz10kN/Bearing3_4":   1,   # Inner Race
}
RESAMPLE_UP, RESAMPLE_DOWN = 15, 32   # 25,600 Hz -> 12,000 Hz

print(f"Using {len(XJTU_BEARING_MAP)} XJTU-SY bearings.")
print(f"Label space matches CWRU: {CLASS_NAMES}")


In [ ]:
#4-2
# Convert every XJTU-SY bearing's raw CSV recordings into spectrograms,
# using the exact same STFT + normalisation pipeline as CWRU (Phase 2),
# so both datasets land in a numerically consistent space.
def xjtu_window_to_spectrogram(segment: np.ndarray) -> np.ndarray:
    t    = torch.tensor(segment.copy(), dtype=torch.float32)
    spec = torch.log(_stft(t) + 1e-8)
    spec = spec.unsqueeze(0).unsqueeze(0)
    spec = F.interpolate(spec, size=(SPEC_SIZE, SPEC_SIZE), mode='bilinear', align_corners=False)
    spec = (spec - spec_mean) / spec_std
    # Per-instance normalisation on top of the global CWRU stats removes
    # any remaining domain-level mean/std mismatch between the two datasets.
    s_mean, s_std = spec.mean(), spec.std()
    spec = (spec - s_mean) / (s_std + 1e-8)
    return spec.squeeze().numpy()


_xjtu_done_flag = os.path.join(XJTU_SPEC_PATH, ".done")

if not os.path.exists(_xjtu_done_flag):
    print("Generating XJTU-SY spectrograms (runs once, several minutes)...")
    for bearing_rel in XJTU_BEARING_MAP:
        src = os.path.join(XJTU_DATASET_PATH, bearing_rel)
        dst = os.path.join(XJTU_SPEC_PATH, bearing_rel)
        os.makedirs(dst, exist_ok=True)
        for fname in tqdm(sorted(os.listdir(src)), desc=bearing_rel, leave=False):
            if not fname.endswith(".csv"): continue
            out_file = os.path.join(dst, fname.replace(".csv", ".npy"))
            if os.path.exists(out_file): continue
            try:
                sig = pd.read_csv(os.path.join(src, fname), header=None)
                sig = pd.to_numeric(sig.iloc[:, 0], errors="coerce").dropna().values
                sig = resample_poly(sig, RESAMPLE_UP, RESAMPLE_DOWN)
                segs  = segment_signal(sig)
                specs = np.array([xjtu_window_to_spectrogram(s) for s in segs])
                np.save(out_file, specs)
            except Exception as e:
                print(f"  Skipped {fname}: {e}")
    open(_xjtu_done_flag, 'w').close()
    print("XJTU-SY spectrogram generation complete.")
else:
    print("XJTU-SY spectrograms already on disk -- skipping.")


In [ ]:
#4-3
# Build the XJTU-SY evaluation set using a conservative 10% lifetime
# cutoff: the first 10% of each bearing's life is labelled Normal, the
# last 10% is labelled with that bearing's fault type, and the ambiguous
# middle 80% (gradual degradation) is discarded entirely.
def collect_xjtu_samples(spec_path, bearing_map, pct=0.10):
    all_specs, all_labels = [], []
    for bearing_rel, fault_label in bearing_map.items():
        out_dir = os.path.join(spec_path, bearing_rel)
        if not os.path.isdir(out_dir): continue
        npy_files = sorted(f for f in os.listdir(out_dir) if f.endswith(".npy"))
        n = len(npy_files); cutoff = max(1, int(n * pct))
        for f in npy_files[:cutoff]:
            s = np.load(os.path.join(out_dir, f))
            all_specs.append(s); all_labels.extend([0] * len(s))
        for f in npy_files[-cutoff:]:
            s = np.load(os.path.join(out_dir, f))
            all_specs.append(s); all_labels.extend([fault_label] * len(s))
    return np.concatenate(all_specs, axis=0), np.array(all_labels)


X_xjtu, y_xjtu = collect_xjtu_samples(XJTU_SPEC_PATH, XJTU_BEARING_MAP)
print(f"XJTU-SY evaluation set: {X_xjtu.shape}")
print(f"Class distribution: {dict(zip(*np.unique(y_xjtu, return_counts=True)))}")


In [ ]:
#4-4
# Zero-shot baseline: apply the CWRU-only model directly to XJTU-SY with
# no retraining at all. This measures the raw size of the domain gap.
xjtu_loader = DataLoader(SpectrogramDataset(X_xjtu, y_xjtu),
                         batch_size=32, shuffle=False, num_workers=2)
acc_x, f1_x, preds_x, targets_x = evaluate(model, xjtu_loader)
print(f"Zero-shot baseline on XJTU-SY: Accuracy={acc_x:.4f}  Macro F1={f1_x:.4f}")
print(classification_report(targets_x, preds_x, target_names=list(CLASS_NAMES.values()), zero_division=0))

cm_x      = confusion_matrix(targets_x, preds_x, labels=[0, 1, 2])
cm_x_norm = cm_x.astype(float) / cm_x.sum(axis=1, keepdims=True) * 100
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
kw = dict(xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values())
sns.heatmap(cm_x,      annot=True, fmt="d",   cmap="Blues",   ax=axes[0], **kw)
sns.heatmap(cm_x_norm, annot=True, fmt=".1f", cmap="Oranges", ax=axes[1], **kw)
for ax, t in zip(axes, ["Counts", "% per row"]):
    ax.set_title(f"Confusion ({t})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle(f"Zero-Shot Transfer: CWRU -> XJTU-SY  |  Acc={acc_x:.1%}  F1={f1_x:.4f}", fontsize=11)
plt.tight_layout()
plt.savefig("conf_xjtu_baseline.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 5 -- Gate: Scalar Features and Difficulty Labels

In [ ]:
#5-0
# Seven cheap scalar statistics computed directly from the raw 1-D signal.
# These are what the gate actually sees -- never the spectrogram, since
# the whole point of the gate is to make its decision before the
# expensive spectrogram computation and CNN forward pass happen.
FEATURE_NAMES = ["RMS", "Kurtosis", "Crest Factor", "Skewness",
                  "Low-band Energy", "High-band Energy", "Spectral Entropy"]
N_FEATURES = len(FEATURE_NAMES)


def extract_features(window: np.ndarray) -> np.ndarray:
    rms          = float(np.sqrt(np.mean(window ** 2)))
    kurt         = float(scipy_kurtosis(window, fisher=True))
    crest_factor = float(np.max(np.abs(window)) / (rms + 1e-8))
    skewness     = float(scipy_skew(window))

    spectrum    = np.abs(np.fft.rfft(window))
    power       = spectrum ** 2
    total_power = power.sum() + 1e-12
    n           = len(spectrum)
    band_low    = float(power[:n // 3].sum() / total_power)
    band_high   = float(power[2 * n // 3:].sum() / total_power)
    psd_norm    = power / total_power
    spec_ent    = float(-np.sum(psd_norm * np.log(psd_norm + 1e-12)))

    return np.array([rms, kurt, crest_factor, skewness,
                      band_low, band_high, spec_ent], dtype=np.float32)


print(f"Feature vector length: {N_FEATURES}")
print("Sample:", dict(zip(FEATURE_NAMES, extract_features(X_train[0]).round(4))))


In [ ]:
#5-1
# Compute these features for every CWRU training window, once.
_feat_path = os.path.join(CWRU_PROC_DIR, "gate_features.npy")

if not os.path.exists(_feat_path):
    print("Extracting scalar features...")
    features = np.array([extract_features(w) for w in tqdm(X_train, desc="Features")])
    np.save(_feat_path, features)
else:
    features = np.load(_feat_path)
    print("Loaded scalar features from disk.")

print(f"Feature matrix: {features.shape}")


In [ ]:
#5-2
# ── Difficulty scoring ────────────────────────────────────────────────
# There's no "how hard is this sample" label anywhere in CWRU, so we make
# one ourselves: corrupt each spectrogram with noise, run it through the
# already-trained baseline model, and measure how confused the model
# gets. A sample the model stays confident on even under heavy noise is
# "easy". A sample where noise flips or muddies the prediction is "hard".
#
# We measure confusion using the entropy of the softmax output, not the
# raw top probability. A well-trained model's raw confidence sits very
# close to 1.0 for almost every sample, so it can't distinguish a mildly
# confident prediction from an extremely confident one. Entropy can,
# since it responds to the whole shape of the probability distribution.
def spectrogram_entropy_score(spec: np.ndarray) -> float:
    """Corrupt one spectrogram with noise N_REPEATS times, run the
    baseline CNN on each noisy copy, and return the average entropy of
    its output. Higher = more confused = harder sample."""
    x = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    sig_power   = x.var().item()
    noise_power = sig_power / (10 ** (SNR_DB / 10))

    entropies = []
    model.eval()
    with torch.no_grad():
        for _ in range(N_REPEATS):
            noisy = x + torch.randn_like(x) * np.sqrt(noise_power)
            probs = torch.softmax(model(noisy), dim=1).squeeze(0)
            entropies.append(-(probs * torch.log(probs + 1e-12)).sum().item())
    return float(np.mean(entropies))


_diff_scores_path = os.path.join(CWRU_PROC_DIR, "difficulty_scores.npy")

if not os.path.exists(_diff_scores_path):
    print(f"Scoring difficulty for {len(X_tr_spec)} windows "
          f"(SNR={SNR_DB}dB, {N_REPEATS} noise draws each)...")
    difficulty_scores = np.array([
        spectrogram_entropy_score(s) for s in tqdm(X_tr_spec, desc="Difficulty")
    ])
    np.save(_diff_scores_path, difficulty_scores)
else:
    difficulty_scores = np.load(_diff_scores_path)
    print("Loaded difficulty scores from disk.")

print(f"\nScore range: min={difficulty_scores.min():.4f}  max={difficulty_scores.max():.4f}")
print("A wide range here means the scores genuinely separate easy from "
      "hard samples. If min and max are nearly equal, the baseline model "
      "is staying confident even under noise and this scoring method "
      "needs a harsher SNR_DB.")


In [ ]:
#5-3
# ── Turn continuous difficulty scores into three discrete tiers ────────
# We split at the 33rd and 66th percentile of the score distribution,
# rather than a fixed number, so the split always produces three usable
# tiers regardless of the exact shape of the distribution. A fixed
# cutoff risks leaving one tier almost empty.
tau_33 = np.percentile(difficulty_scores, 33)
tau_66 = np.percentile(difficulty_scores, 66)

gate_labels = np.where(difficulty_scores <= tau_33, 0,
              np.where(difficulty_scores <= tau_66, 1, 2)).astype(int)
np.save(os.path.join(CWRU_PROC_DIR, "gate_labels.npy"), gate_labels)

print(f"Thresholds: tau_33={tau_33:.4f}  tau_66={tau_66:.4f}")
dist = dict(zip(*np.unique(gate_labels, return_counts=True)))
print("\nTier distribution (should be roughly 33/33/33):")
for tier, count in sorted(dist.items()):
    print(f"  {TIER_NAMES[tier]:8s} -> {count:5d} ({100*count/len(gate_labels):.1f}%)")

print("\nPer fault class:")
for c in sorted(np.unique(y_train)):
    mask = y_train == c
    td = dict(zip(*np.unique(gate_labels[mask], return_counts=True)))
    print(f"  {CLASS_NAMES[c]:12s} Easy={td.get(0,0):4d}  Medium={td.get(1,0):4d}  Hard={td.get(2,0):4d}")


In [ ]:
#5-4
# Plot the difficulty score distribution and the resulting three tiers.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(difficulty_scores, bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(tau_33, color='green', ls='--', label=f'33rd pct ({tau_33:.3f})')
axes[0].axvline(tau_66, color='red',   ls='--', label=f'66th pct ({tau_66:.3f})')
axes[0].set(xlabel='Entropy under noise', ylabel='Count',
            title='Difficulty score distribution')
axes[0].legend(fontsize=8)

classes = sorted(np.unique(y_train))
x = np.arange(len(classes))
width = 0.25
colors = {0: '#55A868', 1: '#4C72B0', 2: '#C44E52'}
for tier in [0, 1, 2]:
    counts = [(gate_labels[y_train == c] == tier).sum() for c in classes]
    axes[1].bar(x + (tier - 1) * width, counts, width, label=TIER_NAMES[tier], color=colors[tier])
axes[1].set_xticks(x); axes[1].set_xticklabels([CLASS_NAMES[c] for c in classes])
axes[1].set_ylabel('Sample count'); axes[1].set_title('Tier distribution per fault class')
axes[1].legend()

plt.suptitle('Three-Tier Difficulty Labelling (Easy / Medium / Hard)', fontsize=12)
plt.tight_layout()
plt.savefig('gate_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## Phase 6 -- Gate: Train the Three-Class GatingMLP

In [ ]:
#6-0
# A tiny MLP: 7 scalar features -> 3 tier logits (Easy / Medium / Hard).
# Deliberately small (fewer than 1,000 parameters) so it's cheap enough
# to run before any spectrogram computation.
class GatingMLP(nn.Module):
    def __init__(self, n_features=N_FEATURES, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes),
        )
    def forward(self, x): return self.net(x)


gate_params = sum(p.numel() for p in GatingMLP().parameters())
print(f"GatingMLP -- {gate_params:,} parameters (Easy / Medium / Hard).")


In [ ]:
#6-1
# Scale features and split into train/validation for the gate.
torch.manual_seed(42); torch.cuda.manual_seed_all(42); np.random.seed(42)

from torch.utils.data import random_split
from sklearn.preprocessing import StandardScaler
import joblib

scaler          = StandardScaler()
features_scaled = scaler.fit_transform(features).astype(np.float32)
joblib.dump(scaler, os.path.join(CWRU_PROC_DIR, "gate_scaler.pkl"))

class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


full_ds = FeatureDataset(features_scaled, gate_labels)
n_val   = int(0.2 * len(full_ds))
n_tr    = len(full_ds) - n_val
g_tr_ds, g_val_ds = random_split(full_ds, [n_tr, n_val],
                                  generator=torch.Generator().manual_seed(42))
g_tr_ldr  = DataLoader(g_tr_ds,  batch_size=256, shuffle=True)
g_val_ldr = DataLoader(g_val_ds, batch_size=256, shuffle=False)
print(f"Gate train: {n_tr}  Gate val: {n_val}")


In [ ]:
#6-2
# Train the gate. Same style of training loop as the baseline model.
def train_gate_epoch(mdl, ldr, opt, crit):
    mdl.train(); total = 0.0
    for x, y in ldr:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(mdl(x), y)
        loss.backward(); opt.step()
        total += loss.item()
    return total / len(ldr)


def evaluate_gate(mdl, ldr):
    mdl.eval(); preds, targets = [], []
    with torch.no_grad():
        for x, y in ldr:
            p = torch.argmax(mdl(x.to(device)), dim=1)
            preds.extend(p.cpu().numpy()); targets.extend(y.numpy())
    return (accuracy_score(targets, preds),
            f1_score(targets, preds, average="macro", zero_division=0),
            preds, targets)


if os.path.exists(GATE_SAVE_PATH):
    print("Gate already trained -- loading from disk.")
    gate_ckpt  = torch.load(GATE_SAVE_PATH, map_location=device)
    gate_model = GatingMLP().to(device)
    gate_model.load_state_dict(gate_ckpt["model_state_dict"])
    gate_model.eval()
    print(f"Loaded. Best F1 = {gate_ckpt['best_f1']:.4f}")
else:
    gate_model     = GatingMLP().to(device)
    gate_criterion = nn.CrossEntropyLoss()
    gate_optimizer = torch.optim.Adam(gate_model.parameters(), lr=1e-3, weight_decay=1e-4)
    gate_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(gate_optimizer, mode="max", factor=0.5, patience=5)

    GATE_EPOCHS, GATE_PATIENCE = 100, 15
    best_gf1, g_patience = 0.0, 0

    print(f"{'Epoch':>6}  {'Loss':>8}  {'Acc':>7}  {'F1':>8}")
    for epoch in range(1, GATE_EPOCHS + 1):
        loss = train_gate_epoch(gate_model, g_tr_ldr, gate_optimizer, gate_criterion)
        acc, f1, _, _ = evaluate_gate(gate_model, g_val_ldr)
        gate_scheduler.step(f1)
        print(f"{epoch:>6}  {loss:>8.4f}  {acc:>7.4f}  {f1:>8.4f}", end="")
        if f1 > best_gf1:
            best_gf1, g_patience = f1, 0
            torch.save({"model_state_dict": gate_model.state_dict(),
                        "best_f1": best_gf1, "n_features": N_FEATURES}, GATE_SAVE_PATH)
            print("  <- best", end="")
        else:
            g_patience += 1
            if g_patience >= GATE_PATIENCE:
                print(f"\nEarly stopping at epoch {epoch}."); break
        print()

    gate_ckpt = torch.load(GATE_SAVE_PATH, map_location=device)
    gate_model.load_state_dict(gate_ckpt["model_state_dict"])
    gate_model.eval()
    print(f"\nBest gate F1 = {best_gf1:.4f}")


In [ ]:
#6-3
# Gate confusion matrix on its held-out validation split.
acc, f1, preds, targets = evaluate_gate(gate_model, g_val_ldr)
print(f"Gate validation: Acc={acc:.4f}  F1={f1:.4f}")

cm      = confusion_matrix(targets, preds, labels=[0, 1, 2])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
kw = dict(xticklabels=[TIER_NAMES[i] for i in range(3)], yticklabels=[TIER_NAMES[i] for i in range(3)])
sns.heatmap(cm,      annot=True, fmt="d",   cmap="Blues",  ax=axes[0], **kw)
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Greens", ax=axes[1], **kw)
for ax, t in zip(axes, ["Counts", "% per row"]):
    ax.set_title(f"Confusion Matrix ({t})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle(f"Gating MLP  \u2014  Acc={acc:.1%}  F1={f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig("gate_confusion.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 7 -- Quantization-Aware Training: Experts A and B

In [ ]:
#7-0
# Quantization configs. Expert A simulates 4-bit precision. Expert B uses
# the standard 8-bit qnnpack config (the ARM-compatible backend needed
# for Raspberry Pi deployment). Expert C's config is defined in Phase 9,
# since it needs a special mixed-precision setup.
from torch.ao.quantization import QConfig, get_default_qat_qconfig, prepare_qat, convert
from torch.ao.quantization.fake_quantize import FakeQuantize
from torch.ao.quantization.observer import MinMaxObserver

torch.backends.quantized.engine = 'qnnpack'  # ARM-compatible backend for the Pi

QCONFIG_A = QConfig(
    activation=FakeQuantize.with_args(observer=MinMaxObserver, quant_min=-8, quant_max=7,
                                       dtype=torch.qint8, qscheme=torch.per_tensor_affine),
    weight=FakeQuantize.with_args(observer=MinMaxObserver, quant_min=-8, quant_max=7,
                                   dtype=torch.qint8, qscheme=torch.per_tensor_symmetric),
)
QCONFIG_B = get_default_qat_qconfig('qnnpack')

print("Quantized backend:", torch.backends.quantized.engine)
print("QCONFIG_A: 4-bit simulated")
print("QCONFIG_B: 8-bit qnnpack")


In [ ]:
#7-1
# Shared QAT training function used by Expert A and Expert B. Both start
# from the trained baseline weights, which speeds up convergence and
# usually gives better final accuracy than starting from scratch.
EXPERT_SAVE = {
    'A': os.path.join(BEARING_DIR, 'expert_A.pth'),
    'B': os.path.join(BEARING_DIR, 'expert_B.pth'),
    'C': os.path.join(BEARING_DIR, 'expert_C.pth'),
}

def train_qat_expert(expert_name, qconfig, X_expert, y_expert, X_val, y_val,
                      epochs=30, patience=5, lr=5e-4, init_state_dict=None):
    """Trains one QAT expert and caches it to disk. If a checkpoint
    already exists, loads it instead of retraining."""
    save_path = EXPERT_SAVE[expert_name]

    m = SpectrogramCNN(num_classes=NUM_CLASSES).to(device)
    m.qconfig = qconfig
    prepare_qat(m, inplace=True)

    if os.path.exists(save_path):
        print(f"Expert {expert_name} already trained -- loading from disk.")
        m.load_state_dict(torch.load(save_path, map_location=device)['model_state_dict'])
        m.eval()
        return m, {}

    if init_state_dict is not None:
        # Careful: prepare_qat() renamed some keys, so load into a plain
        # model first, then copy weights across.
        plain = SpectrogramCNN(num_classes=NUM_CLASSES)
        plain.load_state_dict(init_state_dict)
        m.conv1.weight.data.copy_(plain.conv1.weight.data)
        m.conv1.bias.data.copy_(plain.conv1.bias.data)
        m.conv2.weight.data.copy_(plain.conv2.weight.data)
        m.conv2.bias.data.copy_(plain.conv2.bias.data)
        m.conv3.weight.data.copy_(plain.conv3.weight.data)
        m.conv3.bias.data.copy_(plain.conv3.bias.data)
        m.fc1.weight.data.copy_(plain.fc1.weight.data)
        m.fc1.bias.data.copy_(plain.fc1.bias.data)
        m.fc2.weight.data.copy_(plain.fc2.weight.data)
        m.fc2.bias.data.copy_(plain.fc2.bias.data)
        m.fc3.weight.data.copy_(plain.fc3.weight.data)
        m.fc3.bias.data.copy_(plain.fc3.bias.data)

    _cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_expert)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(_cw, dtype=torch.float32).to(device))
    opt  = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    tr_ldr = DataLoader(SpectrogramDataset(X_expert, y_expert), batch_size=32, shuffle=True, num_workers=2)
    va_ldr = DataLoader(SpectrogramDataset(X_val, y_val), batch_size=64, shuffle=False, num_workers=2)

    best_f1, pat, best_state = 0.0, 0, None
    history = {"loss": [], "acc": [], "f1": []}

    print(f"\n{'Epoch':>6}  {'Loss':>8}  {'Val Acc':>8}  {'Val F1':>8}")
    for ep in range(1, epochs + 1):
        loss = train_one_epoch(m, tr_ldr, opt, crit)
        sch.step()
        acc, f1, _, _ = evaluate(m, va_ldr)
        history["loss"].append(loss); history["acc"].append(acc); history["f1"].append(f1)
        print(f"{ep:>6}  {loss:>8.4f}  {acc:>8.4f}  {f1:>8.4f}", end="")
        if f1 > best_f1:
            best_f1, pat = f1, 0
            best_state = copy.deepcopy(m.state_dict())
            print("  <- best", end="")
        else:
            pat += 1
            if pat >= patience:
                print(f"\nEarly stop at epoch {ep}."); break
        print()

    m.load_state_dict(best_state); m.eval()
    torch.save({'model_state_dict': best_state, 'best_f1': best_f1}, save_path)
    print(f"\nExpert {expert_name} F1 = {best_f1:.4f}  saved.")
    return m, history


import copy
print("QAT training function ready.")


In [ ]:
#7-2
# Both Expert A and Expert B train on the full CWRU training set. They
# differ only in numeric precision -- the gate decides at inference time
# which one a given signal actually needs, not the training data split.
expert_A, hist_A = train_qat_expert('A', QCONFIG_A, X_tr_spec, y_train, X_te_spec, y_test,
                                     epochs=30, patience=5, lr=5e-4,
                                     init_state_dict=model.state_dict())


In [ ]:
#7-3
expert_B, hist_B = train_qat_expert('B', QCONFIG_B, X_tr_spec, y_train, X_te_spec, y_test,
                                     epochs=30, patience=5, lr=5e-4,
                                     init_state_dict=model.state_dict())


## Phase 8 -- Within-XJTU Initialisation Model

In [ ]:
#8-0
# Before training Expert C, we first train a plain full-precision model
# on XJTU-SY alone. Expert C will start from these weights rather than
# from the CWRU baseline, so it begins already adapted to XJTU-SY's
# signal characteristics rather than starting from CWRU-biased filters.
X_xjtu_c_all, y_xjtu_c_all = collect_xjtu_samples(XJTU_SPEC_PATH, XJTU_BEARING_MAP, pct=0.10)
X_xjtu_c, X_xjtu_c_val, y_xjtu_c, y_xjtu_c_val = train_test_split(
    X_xjtu_c_all, y_xjtu_c_all, test_size=0.20, stratify=y_xjtu_c_all, random_state=42)

print(f"XJTU-SY for Expert C -- train: {X_xjtu_c.shape}  val: {X_xjtu_c_val.shape}")
print(f"Train class dist: {dict(zip(*np.unique(y_xjtu_c, return_counts=True)))}")

XJTU_INIT_PATH = os.path.join(BEARING_DIR, 'xjtu_init_model.pth')

if os.path.exists(XJTU_INIT_PATH):
    print("Within-XJTU init model already trained -- loading from disk.")
    xjtu_init_model = SpectrogramCNN(num_classes=NUM_CLASSES).to(device)
    xjtu_init_model.load_state_dict(torch.load(XJTU_INIT_PATH, map_location=device)['model_state_dict'])
    xjtu_init_model.eval()
else:
    _cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_xjtu_c)
    _crit = nn.CrossEntropyLoss(weight=torch.tensor(_cw, dtype=torch.float32).to(device))
    _tr = DataLoader(SpectrogramDataset(X_xjtu_c, y_xjtu_c), batch_size=32, shuffle=True, num_workers=2)
    _va = DataLoader(SpectrogramDataset(X_xjtu_c_val, y_xjtu_c_val), batch_size=32, shuffle=False, num_workers=2)

    xjtu_init_model = SpectrogramCNN(num_classes=NUM_CLASSES).to(device)
    _opt = torch.optim.Adam(xjtu_init_model.parameters(), lr=3e-4, weight_decay=1e-3)
    _sch = torch.optim.lr_scheduler.ReduceLROnPlateau(_opt, mode='max', factor=0.5, patience=4, min_lr=1e-6)

    _best_f1, _pat, _best_state = 0.0, 0, None
    print(f"{'Epoch':>6}  {'Loss':>8}  {'Val F1':>8}")
    for ep in range(1, 61):
        xjtu_init_model.train(); ep_loss = 0.0
        for x, y in _tr:
            x, y = x.to(device), y.to(device)
            _opt.zero_grad()
            loss = _crit(xjtu_init_model(x), y)
            loss.backward(); _opt.step()
            ep_loss += loss.item()
        acc, f1, _, _ = evaluate(xjtu_init_model, _va)
        _sch.step(f1)
        print(f"{ep:>6}  {ep_loss/len(_tr):>8.4f}  {f1:>8.4f}", end="")
        if f1 > _best_f1:
            _best_f1, _pat = f1, 0
            _best_state = copy.deepcopy(xjtu_init_model.state_dict())
            print("  <- best", end="")
        else:
            _pat += 1
            if _pat >= 12:
                print(f"\nEarly stop at epoch {ep}."); break
        print()

    xjtu_init_model.load_state_dict(_best_state); xjtu_init_model.eval()
    torch.save({'model_state_dict': _best_state, 'best_f1': _best_f1}, XJTU_INIT_PATH)
    print(f"\nWithin-XJTU init model F1 = {_best_f1:.4f}  saved.")


## Phase 9 -- Expert C: Mixed Precision, CWRU Hard Tier + XJTU-SY

In [ ]:
#9-0
# Expert C's quantization config. Unlike Expert B, we deliberately
# disable activation quantization after prepare_qat, keeping only the
# weight FakeQuantize active. This gives INT8 weights with full-precision
# (FP32) activations -- genuine mixed precision, not just another 8-bit
# expert with a different training set.
QCONFIG_C = get_default_qat_qconfig('qnnpack')

def apply_expert_C_mixed_precision(m: nn.Module):
    """Disable activation fake-quantization on every layer, leaving only
    the weight fake-quantization active. Must be called right after
    prepare_qat(). This is what makes Expert C mixed precision rather
    than full INT8."""
    for module in m.modules():
        if hasattr(module, 'activation_post_process'):
            module.activation_post_process.disable_fake_quant()
            module.activation_post_process.disable_observer()

print("QCONFIG_C: INT8 weights, FP32 activations (mixed precision).")


In [ ]:
#9-1
# Expert C's training data: the hardest CWRU signals (the Hard tier from
# Phase 5) combined with the full XJTU-SY training split. This lets one
# expert cover both jobs -- hard in-domain cases and fully out-of-domain
# cases -- instead of needing a separate model for each.
hard_mask   = gate_labels == 2
X_cwru_hard = X_tr_spec[hard_mask]
y_cwru_hard = y_train[hard_mask]
print(f"CWRU Hard tier: {X_cwru_hard.shape[0]} samples")
print(f"  Class dist: {dict(zip(*np.unique(y_cwru_hard, return_counts=True)))}")

# Carve out a small validation slice from the Hard tier itself, so we can
# check Expert C's performance on CWRU Hard samples it never trained on.
X_cwru_hard_tr, X_cwru_hard_val, y_cwru_hard_tr, y_cwru_hard_val = train_test_split(
    X_cwru_hard, y_cwru_hard, test_size=0.2, stratify=y_cwru_hard, random_state=42)

X_expert_C_train = np.concatenate([X_cwru_hard_tr, X_xjtu_c],      axis=0)
y_expert_C_train = np.concatenate([y_cwru_hard_tr, y_xjtu_c],      axis=0)
X_expert_C_val   = np.concatenate([X_cwru_hard_val, X_xjtu_c_val], axis=0)
y_expert_C_val   = np.concatenate([y_cwru_hard_val, y_xjtu_c_val], axis=0)

print(f"\nExpert C combined train: {X_expert_C_train.shape[0]} samples "
      f"({dict(zip(*np.unique(y_expert_C_train, return_counts=True)))})")
print(f"Expert C combined val:   {X_expert_C_val.shape[0]} samples "
      f"({dict(zip(*np.unique(y_expert_C_val, return_counts=True)))})")


In [ ]:
#9-2
# Train Expert C: starts from the within-XJTU init model, uses the mixed
# precision QAT config, and trains on the combined CWRU-Hard + XJTU-SY set.
EXPERT_C_SAVE = EXPERT_SAVE['C']

if os.path.exists(EXPERT_C_SAVE):
    print("Expert C already trained -- loading from disk.")
    expert_C = SpectrogramCNN(num_classes=NUM_CLASSES).to(device)
    expert_C.qconfig = QCONFIG_C
    prepare_qat(expert_C, inplace=True)
    apply_expert_C_mixed_precision(expert_C)
    expert_C.load_state_dict(torch.load(EXPERT_C_SAVE, map_location=device)['model_state_dict'])
    expert_C.eval()
    hist_C = {}
    print("Loaded.")
else:
    _cw = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_expert_C_train)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(_cw, dtype=torch.float32).to(device))
    print(f"Expert C training set: {X_expert_C_train.shape}")
    print(f"Class weights: {[round(float(w), 3) for w in _cw]}")

    expert_C = SpectrogramCNN(num_classes=NUM_CLASSES).to(device)
    expert_C.load_state_dict(xjtu_init_model.state_dict())
    expert_C.qconfig = QCONFIG_C
    prepare_qat(expert_C, inplace=True)
    apply_expert_C_mixed_precision(expert_C)
    print("QAT prepared: INT8 weights / FP32 activations.")

    opt = torch.optim.Adam(expert_C.parameters(), lr=1e-4, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=4, min_lr=1e-7)

    tr_ldr = DataLoader(SpectrogramDataset(X_expert_C_train, y_expert_C_train), batch_size=32, shuffle=True, num_workers=2)
    va_ldr = DataLoader(SpectrogramDataset(X_expert_C_val, y_expert_C_val), batch_size=32, shuffle=False, num_workers=2)

    best_f1, pat, best_state = 0.0, 0, None
    hist_C = {"loss": [], "f1": []}

    print(f"\n{'Epoch':>6}  {'Loss':>8}  {'Val F1':>8}")
    for ep in range(1, 41):
        expert_C.train(); ep_loss = 0.0
        for x, y in tr_ldr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(expert_C(x), y)
            loss.backward(); opt.step()
            ep_loss += loss.item()

        acc, f1, _, _ = evaluate(expert_C, va_ldr)
        sch.step(f1)
        hist_C["loss"].append(ep_loss / len(tr_ldr)); hist_C["f1"].append(f1)
        print(f"{ep:>6}  {ep_loss/len(tr_ldr):>8.4f}  {f1:>8.4f}", end="")
        if f1 > best_f1:
            best_f1, pat = f1, 0
            best_state = copy.deepcopy(expert_C.state_dict())
            print("  <- best", end="")
        else:
            pat += 1
            if pat >= 10:
                print(f"\nEarly stop at epoch {ep}."); break
        print()

    expert_C.load_state_dict(best_state); expert_C.eval()
    torch.save({'model_state_dict': best_state, 'best_f1': best_f1}, EXPERT_C_SAVE)
    print(f"\nExpert C F1 = {best_f1:.4f}  saved.")


In [ ]:
#9-3
# Side-by-side confusion matrices for all three experts.
def eval_expert(mdl, X, y):
    ldr = DataLoader(SpectrogramDataset(X, y), batch_size=64, shuffle=False)
    return evaluate(mdl, ldr)

acc_A, f1_A, preds_A, y_A = eval_expert(expert_A, X_te_spec, y_test)
acc_B, f1_B, preds_B, y_B = eval_expert(expert_B, X_te_spec, y_test)
acc_C, f1_C, preds_C, y_C = eval_expert(expert_C, X_expert_C_val, y_expert_C_val)

print(f"Expert A (4-bit)   CWRU test        Acc={acc_A:.4f}  F1={f1_A:.4f}")
print(f"Expert B (8-bit)   CWRU test        Acc={acc_B:.4f}  F1={f1_B:.4f}")
print(f"Expert C (mixed)   CWRU Hard + XJTU Acc={acc_C:.4f}  F1={f1_C:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panels = [
    ("Expert A (4-bit) -- CWRU", y_A, preds_A, acc_A, f1_A),
    ("Expert B (8-bit) -- CWRU", y_B, preds_B, acc_B, f1_B),
    ("Expert C (mixed) -- CWRU Hard + XJTU", y_C, preds_C, acc_C, f1_C),
]
for ax, (title, yt, yp, a, f) in zip(axes, panels):
    cm = confusion_matrix(yt, yp, labels=[0, 1, 2]).astype(float)
    cm = cm / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm, annot=True, fmt=".1f", cmap="Blues",
                xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values(), ax=ax)
    ax.set_title(f"{title}\nAcc={a:.1%}  F1={f:.4f}", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle("Per-Expert Confusion Matrices (% per row)", fontsize=12)
plt.tight_layout()
plt.savefig("expert_conf.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 10 -- Full ApproxMoE System

In [ ]:
#10-0
# We need raw XJTU-SY windows (not spectrograms) so the gate can compute
# its 7 scalar features on them, exactly as it does for CWRU signals.
def collect_xjtu_raw(dataset_path, bearing_map, pct=0.10):
    all_windows, all_labels = [], []
    for bearing_rel, fault_label in bearing_map.items():
        src = os.path.join(dataset_path, bearing_rel)
        if not os.path.isdir(src): continue
        csv_files = sorted(f for f in os.listdir(src) if f.endswith(".csv"))
        n = len(csv_files); cutoff = max(1, int(n * pct))

        for fname, label in ([(f, 0) for f in csv_files[:cutoff]] +
                              [(f, fault_label) for f in csv_files[-cutoff:]]):
            try:
                sig = pd.read_csv(os.path.join(src, fname), header=None)
                sig = pd.to_numeric(sig.iloc[:, 0], errors='coerce').dropna().values
                sig = resample_poly(sig, RESAMPLE_UP, RESAMPLE_DOWN)
                segs = np.array([normalize_window(s) for s in segment_signal(sig)])
                all_windows.extend(segs); all_labels.extend([label] * len(segs))
            except Exception as e:
                print(f"  Skipped {fname}: {e}")
    return np.array(all_windows), np.array(all_labels)


_xjtu_raw_path = os.path.join(CWRU_PROC_DIR, "X_xjtu_raw.npy")
if not os.path.exists(_xjtu_raw_path):
    print("Collecting raw XJTU-SY windows...")
    X_xjtu_raw, y_xjtu_raw = collect_xjtu_raw(XJTU_DATASET_PATH, XJTU_BEARING_MAP)
    np.save(_xjtu_raw_path, X_xjtu_raw)
    np.save(os.path.join(CWRU_PROC_DIR, "y_xjtu_raw.npy"), y_xjtu_raw)
else:
    X_xjtu_raw = np.load(_xjtu_raw_path)
    y_xjtu_raw = np.load(os.path.join(CWRU_PROC_DIR, "y_xjtu_raw.npy"))
    print("Loaded raw XJTU-SY windows from disk.")

print(f"Raw XJTU-SY windows: {X_xjtu_raw.shape}")


In [ ]:
#10-1
# The full routing system:
#   Step 1 -- OOD check: is this signal statistically unlike anything the
#             gate was trained on? If yes, skip straight to Expert C.
#   Step 2 -- Tier decision: the gate predicts Easy / Medium / Hard, and
#             the signal goes to the matching expert (A / B / C).
#   Step 3 -- Confidence cascade: if the assigned expert isn't confident
#             in its own answer, bump the signal up: A -> B -> C.
ESCALATION_THRESHOLD = 0.85
OOD_THRESHOLD        = 10.0

_mahal_mu      = features_scaled.mean(axis=0)
_mahal_cov_inv = np.linalg.pinv(np.cov(features_scaled.T))

def mahal_dist(x_scaled):
    diff = x_scaled - _mahal_mu
    return float(np.sqrt(np.maximum(diff @ _mahal_cov_inv @ diff, 0.0)))


class ApproxMoE:
    def __init__(self, gate, scaler, experts, stft_fn, spec_mean, spec_std,
                 escalation_threshold=ESCALATION_THRESHOLD, ood_threshold=OOD_THRESHOLD):
        self.gate, self.scaler, self.experts = gate, scaler, experts
        self.stft_fn, self.spec_mean, self.spec_std = stft_fn, spec_mean, spec_std
        self.escalation_threshold = escalation_threshold
        self.ood_threshold        = ood_threshold
        self.routing_counts       = {'A': 0, 'B': 0, 'C': 0}
        self.ood_override_count   = 0
        self.escalation_counts    = {'A_to_B': 0, 'B_to_C': 0}

    def _to_spec(self, window):
        w = torch.tensor(window, dtype=torch.float32)
        s = torch.log(self.stft_fn(w) + 1e-8).unsqueeze(0).unsqueeze(0)
        s = F.interpolate(s, size=(SPEC_SIZE, SPEC_SIZE), mode='bilinear', align_corners=False)
        return (s - self.spec_mean) / self.spec_std

    @torch.no_grad()
    def predict_batch(self, raw_windows):
        N = len(raw_windows)
        results = np.zeros(N, dtype=int)

        feats        = np.array([extract_features(w) for w in raw_windows])
        feats_scaled = self.scaler.transform(feats)
        feat_t       = torch.tensor(feats_scaled, dtype=torch.float32).to(device)
        tiers        = torch.argmax(self.gate(feat_t), dim=1).cpu().numpy().copy()

        for i in range(N):
            spec = self._to_spec(raw_windows[i]).to(device)

            # Step 1: the OOD check always wins.
            if mahal_dist(feats_scaled[i]) > self.ood_threshold:
                logits = self.experts['C'](spec)
                self.routing_counts['C'] += 1
                self.ood_override_count += 1
                results[i] = torch.argmax(logits, dim=1).item()
                continue

            # Step 2: tier decision from the gate.
            expert_key = {0: 'A', 1: 'B', 2: 'C'}[tiers[i]]
            if expert_key == 'C':
                logits = self.experts['C'](spec)
                self.routing_counts['C'] += 1
                results[i] = torch.argmax(logits, dim=1).item()
                continue

            # Step 3: confidence cascade.
            logits = self.experts[expert_key](spec)
            conf   = torch.softmax(logits, dim=1).max().item()

            if conf < self.escalation_threshold:
                if expert_key == 'A':
                    self.escalation_counts['A_to_B'] += 1
                    logits = self.experts['B'](spec)
                    conf   = torch.softmax(logits, dim=1).max().item()
                    expert_key = 'B'
                    if conf < self.escalation_threshold:
                        self.escalation_counts['B_to_C'] += 1
                        logits = self.experts['C'](spec)
                        expert_key = 'C'
                elif expert_key == 'B':
                    self.escalation_counts['B_to_C'] += 1
                    logits = self.experts['C'](spec)
                    expert_key = 'C'

            self.routing_counts[expert_key] += 1
            results[i] = torch.argmax(logits, dim=1).item()

        return results

    def routing_summary(self):
        total = sum(self.routing_counts.values())
        print("Routing summary:")
        for k in ['A', 'B', 'C']:
            n = self.routing_counts[k]
            print(f"  Expert {k}: {n:5d}  ({100*n/total:.1f}%)")
        print(f"  OOD overrides: {self.ood_override_count}")
        print(f"  Escalations A->B: {self.escalation_counts['A_to_B']}  "
              f"B->C: {self.escalation_counts['B_to_C']}")


moe_system = ApproxMoE(gate=gate_model, scaler=scaler,
                       experts={'A': expert_A, 'B': expert_B, 'C': expert_C},
                       stft_fn=_stft, spec_mean=spec_mean, spec_std=spec_std)
print("ApproxMoE system built.")


In [ ]:
#10-2
# Evaluate ApproxMoE on the CWRU test set.
BATCH = 256
moe_preds = []
for start in tqdm(range(0, len(X_test), BATCH), desc="ApproxMoE (CWRU)"):
    moe_preds.extend(moe_system.predict_batch(X_test[start:start + BATCH]))
moe_preds = np.array(moe_preds)

moe_acc = accuracy_score(y_test, moe_preds)
moe_f1  = f1_score(y_test, moe_preds, average='macro')
print(f"ApproxMoE on CWRU  Accuracy={moe_acc:.4f}  Macro F1={moe_f1:.4f}")
moe_system.routing_summary()

cm      = confusion_matrix(y_test, moe_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
kw = dict(xticklabels=CLASS_NAMES.values(), yticklabels=CLASS_NAMES.values())
sns.heatmap(cm,      annot=True, fmt="d",   cmap="Blues",  ax=axes[0], **kw)
sns.heatmap(cm_norm, annot=True, fmt=".1f", cmap="Greens", ax=axes[1], **kw)
for ax, t in zip(axes, ["Counts", "% per row"]):
    ax.set_title(f"Confusion Matrix ({t})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.suptitle(f"ApproxMoE on CWRU Test Set  \u2014  Acc={moe_acc:.1%}  F1={moe_f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig("cwru_moe_conf.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
#10-3
# Evaluate ApproxMoE on XJTU-SY. Every signal here should get caught by
# the OOD check and routed to Expert C, since XJTU-SY is a different
# domain than what the gate was trained on.
moe_xjtu_preds = []
for start in tqdm(range(0, len(X_xjtu_raw), BATCH), desc="ApproxMoE (XJTU)"):
    moe_xjtu_preds.extend(moe_system.predict_batch(X_xjtu_raw[start:start + BATCH]))
moe_xjtu_preds = np.array(moe_xjtu_preds)

acc_mx = accuracy_score(y_xjtu_raw, moe_xjtu_preds)
f1_mx  = f1_score(y_xjtu_raw, moe_xjtu_preds, average='macro', zero_division=0)

print(f"ApproxMoE on XJTU-SY   Accuracy={acc_mx:.4f}  Macro F1={f1_mx:.4f}")
print(f"Zero-shot baseline     Accuracy={acc_x:.4f}  Macro F1={f1_x:.4f}")
print(f"Improvement            dAcc={acc_mx-acc_x:+.4f}  dF1={f1_mx-f1_x:+.4f}")
moe_system.routing_summary()


In [ ]:
#10-4
# Final comparison chart: baseline vs ApproxMoE, both datasets.
systems  = ['Baseline CNN\n(FP32)', 'ApproxMoE\n(Expert C = CWRU Hard + XJTU)']
cwru_f1s = [ckpt['best_f1'], moe_f1]
xjtu_f1s = [f1_x, f1_mx]

x_pos = range(len(systems))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar([p - width/2 for p in x_pos], cwru_f1s, width, label='CWRU (in-domain)', color='steelblue', alpha=0.85)
b2 = ax.bar([p + width/2 for p in x_pos], xjtu_f1s, width, label='XJTU-SY (cross-domain)', color='darkorange', alpha=0.85)
for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xticks(list(x_pos)); ax.set_xticklabels(systems, fontsize=10)
ax.set_ylabel('Macro F1'); ax.set_ylim(0, 1.12)
ax.set_title('Baseline CNN vs ApproxMoE\nCWRU (in-domain) and XJTU-SY (cross-domain)')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('final_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'System':<45} {'CWRU F1':>9} {'XJTU F1':>9}")
print('=' * 65)
for name, cf, xf in zip(systems, cwru_f1s, xjtu_f1s):
    print(f"{name.replace(chr(10), ' '):<45} {cf:>9.4f} {xf:>9.4f}")
